<a href="https://colab.research.google.com/github/febriyansyah-id/COLLAB/blob/main/notebooks/NLP-assignment-P05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Assignment P05
## Information Retrieval

Notebook ini menjalankan preprocessing, inverted index, Boolean retrieval, TF-IDF ranking, cosine similarity, dan evaluasi precision-recall di Google Colab.

## Konsep utama

- **Inverted index** memetakan term ke daftar `document_id`.
- **Boolean retrieval** menggunakan operasi AND, OR, dan NOT.
- **TF-IDF** memberi bobot pada term berdasarkan kepentingannya.
- **Cosine similarity** mengurutkan dokumen berdasarkan kedekatan vektor.
- **Precision** mengukur ketepatan hasil, sedangkan **recall** mengukur cakupan dokumen relevan.

## Deskripsi library dan model

| Komponen | Deskripsi |
|---|---|
| NumPy | Operasi array numerik dan pengurutan skor similarity. |
| pandas | Menampilkan corpus, inverted index, ranking, dan tabel evaluasi. |
| re | Tokenisasi dan normalisasi pola teks menggunakan regular expression. |
| scikit-learn | Menyediakan `TfidfVectorizer` dan `cosine_similarity`. |
| Inverted index | Struktur index lexical yang memetakan term ke postings list dokumen. |
| Boolean retrieval | Model retrieval berbasis logika AND, OR, dan NOT. |
| TF-IDF | Model sparse yang memberi bobot pada term berdasarkan frekuensi dan kelangkaannya. |
| Cosine similarity | Ukuran kemiripan arah antara vektor query dan vektor dokumen. |
| BM25 | Model probabilistic ranking yang memperhitungkan TF, IDF, saturasi term, dan panjang dokumen. |
| Dense retrieval | Model yang membandingkan embedding query dan dokumen untuk menangkap kemiripan semantik. |
| Hybrid retrieval | Gabungan lexical retrieval seperti BM25 dan dense retrieval. |

In [1]:
%pip install -q numpy pandas scikit-learn matplotlib seaborn

In [2]:
import re
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Library IR siap digunakan.')

Library IR siap digunakan.


## 1. Corpus dan preprocessing

In [3]:
documents = {
    1: 'Python digunakan untuk analisis data dan machine learning.',
    2: 'Information retrieval mencari dokumen yang relevan dengan query.',
    3: 'Machine learning dapat digunakan untuk klasifikasi teks.',
    4: 'Search engine menggunakan indexing dan ranking dokumen.',
    5: 'TF-IDF memberikan bobot pada kata yang penting dalam dokumen.',
    6: 'Inverted index menyimpan daftar dokumen untuk setiap term.',
    7: 'Semantic search menggunakan embedding untuk memahami makna.',
    8: 'Natural language processing memproses data berupa teks.',
    9: 'Cosine similarity membandingkan kemiripan vektor dokumen.',
    10: 'BM25 dapat digunakan untuk ranking pada sistem retrieval.',
}

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

tokenized = {doc_id: tokenize(text) for doc_id, text in documents.items()}
pd.DataFrame({'document_id': documents.keys(), 'text': documents.values(), 'tokens': tokenized.values()})

,document_id,text,tokens
0,1,Python digunakan untuk analisis data dan machi...,"[python, digunakan, untuk, analisis, data, dan..."
1,2,Information retrieval mencari dokumen yang rel...,"[information, retrieval, mencari, dokumen, yan..."
2,3,Machine learning dapat digunakan untuk klasifi...,"[machine, learning, dapat, digunakan, untuk, k..."
3,4,Search engine menggunakan indexing dan ranking...,"[search, engine, menggunakan, indexing, dan, r..."
4,5,TF-IDF memberikan bobot pada kata yang penting...,"[tf, idf, memberikan, bobot, pada, kata, yang,..."
5,6,Inverted index menyimpan daftar dokumen untuk ...,"[inverted, index, menyimpan, daftar, dokumen, ..."
6,7,Semantic search menggunakan embedding untuk me...,"[semantic, search, menggunakan, embedding, unt..."
7,8,Natural language processing memproses data ber...,"[natural, language, processing, memproses, dat..."
8,9,Cosine similarity membandingkan kemiripan vekt...,"[cosine, similarity, membandingkan, kemiripan,..."
9,10,BM25 dapat digunakan untuk ranking pada sistem...,"[bm25, dapat, digunakan, untuk, ranking, pada,..."


## 2. Inverted index

Index menyimpan setiap term satu kali per dokumen, kemudian mencatat dokumen yang memuat term tersebut.

In [4]:
def build_inverted_index(documents):
    index = defaultdict(set)
    for doc_id, text in documents.items():
        for term in set(tokenize(text)):
            index[term].add(doc_id)
    return {term: sorted(doc_ids) for term, doc_ids in index.items()}

index = build_inverted_index(documents)
index_table = pd.DataFrame({
    'term': list(index.keys()),
    'postings': list(index.values()),
}).sort_values('term').reset_index(drop=True)
display(index_table.head(15))
print('Jumlah term dalam vocabulary:', len(index))

,term,postings
0,analisis,[1]
1,berupa,[8]
2,bm25,[10]
3,bobot,[5]
4,cosine,[9]
5,daftar,[6]
6,dalam,[5]
7,dan,"[1, 4]"
8,dapat,"[3, 10]"
9,data,"[1, 8]"


Jumlah term dalam vocabulary: 54


## 3. Boolean retrieval

In [5]:
def boolean_and(index, terms):
    sets = [set(index.get(term, [])) for term in terms]
    return set.intersection(*sets) if sets else set()

def boolean_or(index, terms):
    result = set()
    for term in terms:
        result.update(index.get(term, []))
    return result

def boolean_not(index, term, universe):
    return set(universe) - set(index.get(term, []))

print('machine AND learning:', sorted(boolean_and(index, ['machine', 'learning'])))
print('retrieval OR indexing:', sorted(boolean_or(index, ['retrieval', 'indexing'])))
print('NOT classification:', sorted(boolean_not(index, 'classification', documents)))

machine AND learning: [1, 3]
retrieval OR indexing: [2, 4, 10]
NOT classification: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## 4. Ranked retrieval dengan TF-IDF

In [6]:
texts = [documents[doc_id] for doc_id in sorted(documents)]
vectorizer = TfidfVectorizer(tokenizer=tokenize, token_pattern=None)
X = vectorizer.fit_transform(texts)

def rank_query(query, top_k=5):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, X).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return pd.DataFrame({
        'document_id': [sorted(documents)[i] for i in order],
        'score': scores[order],
        'text': [texts[i] for i in order],
    })

query = 'information retrieval'
ranking = rank_query(query)
print('Ranking untuk query:', query)
display(ranking.style.format({'score': '{:.4f}'}))

Ranking untuk query: information retrieval


,document_id,score,text
0,2,0.5034,Information retrieval mencari dokumen yang relevan dengan query.
1,10,0.2287,BM25 dapat digunakan untuk ranking pada sistem retrieval.
2,9,0.0000,Cosine similarity membandingkan kemiripan vektor dokumen.
3,8,0.0000,Natural language processing memproses data berupa teks.
4,6,0.0000,Inverted index menyimpan daftar dokumen untuk setiap term.


## 5. Evaluasi precision dan recall

In [7]:
relevant = {
    'machine learning': {1, 3},
    'information retrieval': {2, 4, 6, 10},
    'semantic search': {7, 9},
}

def precision_recall_at_k(retrieved, gold, k):
    actual = set(retrieved[:k])
    hits = len(actual & gold)
    precision = hits / k if k else 0.0
    recall = hits / len(gold) if gold else 0.0
    return precision, recall

evaluation = []
for query_text, gold in relevant.items():
    retrieved = rank_query(query_text, top_k=5)['document_id'].tolist()
    precision, recall = precision_recall_at_k(retrieved, gold, 5)
    evaluation.append({
        'query': query_text, 'retrieved': retrieved,
        'precision@5': precision, 'recall@5': recall,
    })

evaluation_table = pd.DataFrame(evaluation)
display(evaluation_table)

,query,retrieved,precision@5,recall@5
0,machine learning,"[3, 1, 9, 10, 8]",0.4,1.00
1,information retrieval,"[2, 10, 9, 8, 6]",0.6,0.75
2,semantic search,"[7, 4, 9, 10, 6]",0.4,1.00


In [8]:
assert X.shape[0] == len(documents)
assert len(index) > 0
assert evaluation_table['precision@5'].between(0, 1).all()
assert evaluation_table['recall@5'].between(0, 1).all()
print('Validasi IR: PASS')

Validasi IR: PASS


## 6. Analisis dan kesimpulan otomatis

In [9]:
mean_precision = evaluation_table['precision@5'].mean()
mean_recall = evaluation_table['recall@5'].mean()
top_result = ranking.iloc[0]
top_three = ranking.head(3)[['document_id', 'score']].to_dict('records')

analysis = f'''
ANALISIS OTOMATIS
Corpus berisi {len(documents)} dokumen dan inverted index memiliki {len(index)} term unik.
Boolean retrieval mengembalikan himpunan dokumen tanpa ranking, sedangkan
TF-IDF dan cosine similarity mengurutkan dokumen berdasarkan relevansi lexical.
Untuk query '{query}', dokumen teratas adalah dokumen {int(top_result['document_id'])}
dengan skor {top_result['score']:.4f}. Rata-rata precision@5 adalah {mean_precision:.3f}
dan rata-rata recall@5 adalah {mean_recall:.3f}.
Tiga hasil teratas: {top_three}.
Boolean retrieval tepat untuk filter lexical eksplisit, TF-IDF sesuai sebagai
baseline ranked retrieval, sedangkan BM25 atau dense retrieval dapat diuji
untuk corpus dan kebutuhan semantik yang lebih kompleks.
'''

conclusion = f'''
KESIMPULAN OTOMATIS
Inverted index berhasil memetakan term ke postings list dan mendukung
Boolean retrieval. Ranked retrieval berbasis TF-IDF memberikan urutan hasil
yang lebih informatif menggunakan cosine similarity. Library utama yang
digunakan adalah NumPy untuk numerik, pandas untuk tabel, regular expression
untuk tokenisasi, dan scikit-learn untuk TF-IDF serta cosine similarity.
Sistem ini cocok sebagai
baseline sparse retrieval, tetapi belum memahami sinonim dan parafrase.
Pengembangan berikutnya dapat membandingkan BM25, dense retrieval, dan hybrid
retrieval pada corpus yang lebih besar dengan relevance judgment yang lebih lengkap.
'''
print(analysis)
print(conclusion)


ANALISIS OTOMATIS
Corpus berisi 10 dokumen dan inverted index memiliki 54 term unik.
Boolean retrieval mengembalikan himpunan dokumen tanpa ranking, sedangkan
TF-IDF dan cosine similarity mengurutkan dokumen berdasarkan relevansi lexical.
Untuk query 'information retrieval', dokumen teratas adalah dokumen 2
dengan skor 0.5034. Rata-rata precision@5 adalah 0.467
dan rata-rata recall@5 adalah 0.917.
Tiga hasil teratas: [{'document_id': 2, 'score': 0.5033971218221373}, {'document_id': 10, 'score': 0.2286934144439168}, {'document_id': 9, 'score': 0.0}].
Boolean retrieval tepat untuk filter lexical eksplisit, TF-IDF sesuai sebagai
baseline ranked retrieval, sedangkan BM25 atau dense retrieval dapat diuji
untuk corpus dan kebutuhan semantik yang lebih kompleks.


KESIMPULAN OTOMATIS
Inverted index berhasil memetakan term ke postings list dan mendukung
Boolean retrieval. Ranked retrieval berbasis TF-IDF memberikan urutan hasil
yang lebih informatif menggunakan cosine similarity. Library utam